In [1]:
import pandas as pd, numpy as np

In [2]:
df_simule = pd.read_csv(r"../bronze/dataset_brut.csv")
SEND_MESSAGE_INTERVAL_IN_SECONDES = 30 # 30 secondes

In [3]:
df_simule.head()

,timestamp,machine_id,type_machine,secteur,type_metal,vitesse_rotation_nominal,courant_moteur_nominal,pression_hydraulique_nominal,statut_nominal,temp_base_moteur,...,iot_vitesse_rotation,iot_courant_moteur,iot_pression_hydraulique,iot_temperature,iot_vibration_rms,iot_vibration_peak,iot_charge_moteur,age_jours,age_virtuel_jours,RUL_jours
0,2026-06-01 00:00:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.815780,0.000000,48.711109,0.856741,1.267143,45.0,1200.000000,840.000000,1.513542
1,2026-06-01 00:00:30,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.028414,0.000000,48.552522,0.852020,1.355303,45.0,1200.000347,840.000347,1.513194
2,2026-06-01 00:01:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.318093,0.355548,48.749168,0.849968,1.226547,45.0,1200.000694,840.000694,1.512847
3,2026-06-01 00:01:30,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.882224,0.000000,48.968161,0.881784,1.205064,45.0,1200.001042,840.001042,1.512500
4,2026-06-01 00:02:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.984053,0.240067,48.529023,0.837604,1.358074,45.0,1200.001389,840.001389,1.512153


In [4]:
df_simule.info()

<class 'pandas.DataFrame'>
RangeIndex: 1854720 entries, 0 to 1854719
Data columns (total 22 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   timestamp                     str    
 1   machine_id                    str    
 2   type_machine                  str    
 3   secteur                       str    
 4   type_metal                    str    
 5   vitesse_rotation_nominal      int64  
 6   courant_moteur_nominal        float64
 7   pression_hydraulique_nominal  float64
 8   statut_nominal                str    
 9   temp_base_moteur              float64
 10  iot_statut_machine            str    
 11  label_gmao                    str    
 12  iot_vitesse_rotation          float64
 13  iot_courant_moteur            float64
 14  iot_pression_hydraulique      float64
 15  iot_temperature               float64
 16  iot_vibration_rms             float64
 17  iot_vibration_peak            float64
 18  iot_charge_moteur             flo

In [5]:
df_clean_iot = df_simule.drop(columns=[
    "age_jours",
    "age_virtuel_jours",
    "label_gmao",
    "RUL_jours",
    "secteur",
    "type_machine",
    "vitesse_rotation_nominal",
    "courant_moteur_nominal",
    "pression_hydraulique_nominal",
    "statut_nominal",
    "type_metal",
    "temp_base_moteur",
    "iot_statut_machine"
])
df_clean_plc = df_simule[[
        "machine_id",
        "timestamp",
        "iot_statut_machine",
        "type_metal", # TODO: mettre un id et non un str
]]

In [6]:
df_clean_iot["timestamp"] = pd.to_datetime(
    df_clean_iot["timestamp"],
    format="%Y-%m-%d %H:%M:%S"
)
df_clean_plc["timestamp"] = pd.to_datetime(
    df_clean_plc["timestamp"],
    format="%Y-%m-%d %H:%M:%S"
)
df_clean_iot.info()

<class 'pandas.DataFrame'>
RangeIndex: 1854720 entries, 0 to 1854719
Data columns (total 9 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   timestamp                 datetime64[us]
 1   machine_id                str           
 2   iot_vitesse_rotation      float64       
 3   iot_courant_moteur        float64       
 4   iot_pression_hydraulique  float64       
 5   iot_temperature           float64       
 6   iot_vibration_rms         float64       
 7   iot_vibration_peak        float64       
 8   iot_charge_moteur         float64       
dtypes: datetime64[us](1), float64(7), str(1)
memory usage: 127.4 MB


In [7]:
df_clean_iot = (
    df_clean_iot.sort_values("timestamp")
      .reset_index(drop=True)
)
df_clean_plc = (
    df_clean_plc.sort_values("timestamp")
      .reset_index(drop=True)
)

In [8]:
pd.set_option("display.max_colwidth", None)
df_clean_iot.head(18)

,timestamp,machine_id,iot_vitesse_rotation,iot_courant_moteur,iot_pression_hydraulique,iot_temperature,iot_vibration_rms,iot_vibration_peak,iot_charge_moteur
0,2026-06-01 00:00:00,MCH-001,6000.0,19.815780,0.000000,48.711109,0.856741,1.267143,45.0
1,2026-06-01 00:00:00,MCH-002,4500.0,22.085784,0.000000,40.794459,0.856255,1.257802,45.0
2,2026-06-01 00:00:00,MCH-007,4800.0,16.279980,0.258873,39.129279,0.435057,0.508685,45.0
3,2026-06-01 00:00:00,MCH-004,1250.0,43.193466,0.474612,56.080133,0.640334,1.086669,80.0
4,2026-06-01 00:00:00,MCH-006,0.0,54.966129,180.021158,61.644770,2.088440,3.206393,70.0
5,2026-06-01 00:00:00,MCH-005,4300.0,24.147071,0.000000,43.365719,0.835820,1.445014,45.0
6,2026-06-01 00:00:00,MCH-003,5000.0,14.950068,0.000000,40.556213,0.415316,0.663997,45.0
7,2026-06-01 00:00:30,MCH-007,4800.0,15.856575,0.000000,38.853727,0.418200,0.633906,45.0
8,2026-06-01 00:00:30,MCH-006,0.0,55.047077,180.699071,61.560815,2.145707,3.266227,70.0
9,2026-06-01 00:00:30,MCH-002,4500.0,21.962499,0.160821,40.963952,0.824503,1.247033,45.0


In [9]:
df_clean_plc.head(18)  # TODO: mettre un numéro à iot_statut_machine

,machine_id,timestamp,iot_statut_machine,type_metal
0,MCH-001,2026-06-01 00:00:00,Production,Aluminium
1,MCH-002,2026-06-01 00:00:00,Production,Aluminium
2,MCH-007,2026-06-01 00:00:00,Production,Aluminium
3,MCH-004,2026-06-01 00:00:00,Production,Titane
4,MCH-006,2026-06-01 00:00:00,Production,Acier
5,MCH-005,2026-06-01 00:00:00,Production,Aluminium
6,MCH-003,2026-06-01 00:00:00,Production,Aluminium
7,MCH-007,2026-06-01 00:00:30,Production,Aluminium
8,MCH-006,2026-06-01 00:00:30,Production,Acier
9,MCH-002,2026-06-01 00:00:30,Production,Aluminium


In [11]:
df_clean_iot.to_csv(f"./dataset_iot.csv", index=False, encoding='utf-8')
df_clean_plc.to_csv(f"./dataset_plc.csv", index=False, encoding='utf-8')